# 1. Configuração do Ambiente e Importação de Dependências

Nesta etapa inicial, preparamos o ambiente de execução importando as bibliotecas fundamentais. Utilizaremos o **Pandas** e o **NumPy** para a estruturação e manipulação matricial dos dados. A biblioteca **re (Regex)** será essencial para extrair valores numéricos de strings formatadas. Por fim, como o script é executado no Google Colab, montaremos o Google Drive para permitir o acesso direto e a escrita dos arquivos na nuvem.

In [32]:
import pandas as pd
import re
import numpy as np
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# 2. Módulo de Carregamento e Padronização Inicial

A função `carregar_dados_geoquimicos` é responsável pela ingestão segura dos dados brutos. Para evitar problemas comuns na manipulação de DataFrames, ela aplica uma limpeza inicial nos cabeçalhos (`columns`), removendo espaços em branco residuais (trailing/leading whitespaces). A leitura é feita considerando as formatações originais do Excel para não corromper identificadores de censura.

In [33]:
def carregar_dados_geoquimicos(caminho_arquivo, aba=0):
    try:
        # Carregamos o Excel garantindo que os dados sejam lidos sem conversões agressivas
        # O dtype=str pode ser usado se o Excel tiver formatações que confundam o Pandas
        df = pd.read_excel(caminho_arquivo, sheet_name=aba)

        # Removemos possíveis espaços em branco nos nomes das colunas
        df.columns = [col.strip() for col in df.columns]

        return df
    except Exception as e:
        print(f"Erro ao carregar o arquivo: {e}")
        return None

# Exemplo de uso encadeado:
# dados_brutos = carregar_dados_geoquimicos("projeto_geoquimico.xlsx")
# if dados_brutos is not None:
#     dados_tratados, descartados = tratar_dados_geoquimicos(dados_brutos)

# 3. Módulo de Tratamento Estatístico e Imputação (LOD)

Este é o núcleo analítico do pipeline. A função `tratar_dados_geoquimicos` processa as leituras censuradas (valores marcados com `<`)[cite: 1] através de uma abordagem híbrida, adaptando-se dinamicamente ao formato em que a censura foi registrada. A lógica central opera sob as seguintes regras de negócio:

*   **Avaliação de Censura:** Calcula a proporção de dados censurados por coluna (elemento)[cite: 1].
*   **Critério de Descarte Estrito:** Colunas com mais de 30% de leituras censuradas são removidas da matriz de dados[cite: 1].
*   **Critério de Imputação Dinâmica (LOD):** A função avalia o conteúdo que acompanha o símbolo de censura para decidir a ação:
    *   **Com Valor Numérico (ex: `<0,5` ou `<20.0`):** O script extrai o valor numérico do Limite de Detecção (LOD)[cite: 1], lidando automaticamente com separadores de ponto ou vírgula. A imputação é feita com base na distribuição: se a censura for menor que 10%, os valores ausentes recebem 66% do LOD[cite: 1]. Se a censura estiver entre 10% e 30%, recebem 33% do LOD[cite: 1].
    *   **Sem Valor Numérico (ex: `<LD`, `<LOD`):** Quando a censura é apenas qualitativa, a etapa de cálculo de proporção (66%/33%) é ignorada de forma segura. Os valores são transformados em marcadores de dados ausentes (`NaN`), permitindo que a coluna sobreviva sem quebrar futuras operações matemáticas.
*   **Conversão de Tipo:** Garante que todo o DataFrame resultante contenha apenas valores numéricos computáveis[cite: 1].

In [34]:
import pandas as pd
import re

def tratar_dados_geoquimicos(df):
    df_limpo = df.copy()
    colunas_para_descartar = []

    for coluna in df.columns:
        # Identifica células que contêm o caractere de censura '<'
        mascara_lod = df[coluna].astype(str).str.contains('<')

        total_amostras = len(df)
        total_abaixo_lod = mascara_lod.sum()
        proporcao_censura = total_abaixo_lod / total_amostras

        # Se houver valores censurados (com '<')
        if total_abaixo_lod > 0:
            if proporcao_censura > 0.30:
                # Regra Geral: Descarta se a censura for maior que 30%
                colunas_para_descartar.append(coluna)
            else:
                # Pega o primeiro valor censurado da coluna como amostra
                exemplo_lod = df.loc[mascara_lod, coluna].iloc[0]

                # Regex atualizada para capturar números com ponto ou vírgula (ex: '0.5' ou '0,5')
                matches = re.findall(r"[-+]?\d*[\.,]?\d+", str(exemplo_lod))

                # Converte a coluna para numérico (valores como '<0,5' ou '<LD' viram NaN)
                df_limpo[coluna] = pd.to_numeric(df_limpo[coluna], errors='coerce')

                if matches:
                    # CASO 1: Existe um valor numérico após o '<' (ex: '<0,5')
                    # Substitui a vírgula por ponto para o Python não dar erro no float()
                    lod_num = float(matches[0].replace(',', '.'))

                    # Aplica a regra de imputação de 66% ou 33%
                    fator = 0.66 if proporcao_censura < 0.10 else 0.33
                    df_limpo[coluna] = df_limpo[coluna].fillna(lod_num * fator)
                else:
                    # CASO 2: NÃO existe valor numérico (ex: '<LD', '<LOD')
                    # O Pandas já transformou em NaN no to_numeric.
                    # Apenas ignoramos a imputação e mantemos os dados passíveis de cálculo.
                    pass
        else:
            # Colunas sem '<' são convertidas para numérico normalmente
            df_limpo[coluna] = pd.to_numeric(df_limpo[coluna], errors='coerce')

    # Remove as colunas marcadas
    df_limpo = df_limpo.drop(columns=colunas_para_descartar)
    return df_limpo, colunas_para_descartar

# 4. Módulo de Exportação e Auditoria

Após o tratamento, precisamos persistir os resultados. A função `salvar_dados_tratados` consolida os dados limpos e gera um novo arquivo Excel. Para manter a rastreabilidade do pipeline, o arquivo é gerado com duas abas (`sheets`): a primeira contendo a matriz de dados processada e a segunda (`Log_Remocao`) registrando especificamente quais elementos foram descartados pelo critério de excesso de censura.

In [35]:
def salvar_dados_tratados(df, colunas_descartadas, caminho_saida="dados_geoquimicos_processados.xlsx"):
    try:
        # Criamos um escritor do Excel para salvar múltiplas abas se necessário
        with pd.ExcelWriter(caminho_saida, engine='openpyxl') as writer:
            # Salva os dados principais
            df.to_excel(writer, sheet_name='Dados_Tratados', index=True)

            # Cria uma aba secundária com o relatório de colunas descartadas
            if colunas_descartadas:
                df_descarte = pd.DataFrame(colunas_descartadas, columns=['Elementos_Removidos_mais_30pct_LOD'])
                df_descarte.to_excel(writer, sheet_name='Log_Remocao', index=False)

        print(f"Arquivo salvo com sucesso em: {caminho_saida}")
    except Exception as e:
        print(f"Erro ao salvar o arquivo: {e}")

# 5. Fluxo Principal de Execução

A função `executar_pipeline_geoquimico` orquestra a sequência completa de operações. É nesta etapa final que fornecemos o caminho do arquivo de entrada hospedado no Drive. O fluxo consolida o carregamento, o tratamento do Limite de Detecção (LOD) e a exportação, emitindo logs no console sobre o status da execução e informando quais colunas foram eliminadas.

In [36]:


# --- FLUXO PRINCIPAL DE EXECUÇÃO ---

def executar_pipeline_geoquimico(arquivo_entrada):
    nome_saida = "resultado_geoquimico_tratado.xlsx"

    # 1. Carregamento
    print("Iniciando carga de dados...")
    dados_brutos = carregar_dados_geoquimicos(arquivo_entrada)

    if dados_brutos is not None:
        # 2. Tratamento e Imputação
        print("Aplicando filtros de Limite de Detecção (LOD)...")
        dados_tratados, excluidos = tratar_dados_geoquimicos(dados_brutos)

        # 3. Exportação
        print(f"Colunas removidas por excesso de censura (>30%): {excluidos}")
        salvar_dados_tratados(dados_tratados, excluidos, nome_saida)

# Para rodar o código, basta passar o nome do seu arquivo excel:


In [37]:
executar_pipeline_geoquimico("/content/drive/MyDrive/geoquimicos/P19_resultadosFRXp.xlsx")

Iniciando carga de dados...
Aplicando filtros de Limite de Detecção (LOD)...
Colunas removidas por excesso de censura (>30%): ['V', 'Cr', 'Co', 'Cu', 'Al']
Arquivo salvo com sucesso em: resultado_geoquimico_tratado.xlsx


In [38]:
dados_brutos = carregar_dados_geoquimicos("/content/drive/MyDrive/geoquimicos/P2_resultadosFRXp.xlsx")
if dados_brutos is not None:
     dados_tratados, descartados = tratar_dados_geoquimicos(dados_brutos)